# AI FinOps - Gestor de Base de Datos SQLite

Usa este notebook para consultar, auditar y modificar de manera cómoda los datos de consumo de la base de datos SQLite (`finops_db.sqlite`).

In [ ]:
import sqlite3
import os

# Ruta al archivo SQLite de la aplicación
db_path = os.path.join('src', 'app', 'finops_db.sqlite')
print(f"Base de datos: {db_path}")
print(f"Existe el archivo: {'Sí' if os.path.exists(db_path) else 'No'}")

### Función auxiliar para consultas
Esta función ejecuta sentencias SQL. Si es un `SELECT`, devuelve un DataFrame de `pandas` (si está instalado) o una lista de diccionarios.

In [ ]:
def run_query(query, params=()):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    try:
        cursor.execute(query, params)
        if query.strip().upper().startswith("SELECT"):
            rows = cursor.fetchall()
            columns = [col[0] for col in cursor.description]
            data = [dict(r) for r in rows]
            try:
                import pandas as pd
                return pd.DataFrame(data, columns=columns)
            except ImportError:
                return data
        else:
            conn.commit()
            print(f"Operación exitosa. Filas afectadas: {cursor.rowcount}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()

### 1. Consultar Estado de Consumidores (Presupuestos)

In [ ]:
run_query("SELECT * FROM consumers")

### 2. Consultar Historial de Transacciones (Últimas 10)

In [ ]:
run_query("SELECT * FROM transactions ORDER BY timestamp DESC LIMIT 10")

### 3. Ver Alertas Disparadas

In [ ]:
run_query("SELECT * FROM alerts")

### 4. Modificar Límites de Presupuesto
Puedes cambiar el presupuesto asignado a cualquier consumidor modificando el campo `budget_limit`.

In [ ]:
# Cambiar límite de equipo-marketing a 15.0 dólares
run_query("UPDATE consumers SET budget_limit = 15.0 WHERE id = 'equipo-marketing'")

# Comprobar el cambio
run_query("SELECT * FROM consumers WHERE id = 'equipo-marketing'")

### 5. Restablecer el Gasto Acumulado y Alertas de un Equipo
Si un equipo ha bloqueado sus peticiones por superar el presupuesto y deseas resetearlo de forma aislada:

In [ ]:
target_team = 'equipo-marketing'

# Restablecer gasto y bandera de alerta
run_query("UPDATE consumers SET spent = 0.0, alert_fired = 0 WHERE id = ?", (target_team,))

# Eliminar sus alertas del registro
run_query("DELETE FROM alerts WHERE consumer_id = ?", (target_team,))

# Verificar
run_query("SELECT * FROM consumers WHERE id = ?", (target_team,))